# England-wide A-level model, and the best schools for chosen subjects in an area

One model for all of England, then a read-out for one area. The model is fitted on **all 2,747 institutions** with GCSE or A-level results, in every region. It gives each institution an estimate of its *true* A-level value added in each subject, with an interval, and the probability that it is among the best in the chosen area (London by default) and in England. The area is only a filter applied at the end, set in one cell below; nothing in the fit is specific to it.

**What is in the model**

- **GCSE side** (schools with Progress 8 results): general quality $g_i$, the Maths-and-Science versus English-and-Open tilt $h_i$, and consistency $s_i$, all with known measurement error, as in the earlier notebooks.
- **A-level side** (all institutions): nine subject groups, each with its own slopes on $g_i$ and $h_i$, a shared A-level quality $u_i$ that GCSE does not explain, and scatter specific to the group. Maths, Economics and Psychology are separate groups; the rest are Sciences, English, Humanities, Other social sciences (Sociology, Politics, Law), Business & Computing and Creative arts.
- **Geography at two levels.** Regions (9) and local authorities (152) each shift general GCSE quality, and each shifts the shared A-level quality. The local-authority layer matters: differences between authorities within a region are about as large as those between regions.
- **Institutions with A-level results but no GCSE results** (864: mostly independent schools and colleges) are in the model. They are not given an invented GCSE profile: they get the typical GCSE quality for where they are, no tilt, their own scatter, and a separate mean shift by type (independent school, college, other). An earlier version that gave them a random GCSE profile inflated the GCSE slopes for everyone (35% of Maths variance explained instead of 29%).
- **School names** come from the public "Get Information About Schools" register (`data/school-names.csv`, matched on URN).

**How to read the results.** A school's *estimate* is what the model believes its true value added is, after allowing for cohort size (small cohorts are pulled toward what the rest of the evidence supports). **P(top fifth)** is the probability that its true value added is among the top 20% of institutions offering that subject, in the area or in England. Read this, not the rank: many institutions are statistically indistinguishable. Value added is a school average for the pupils who took the subject there, for one year; it says nothing certain about a particular pupil.

In [ ]:
import arviz as az
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymc as pm
import pytensor.tensor as pt
import xarray as xr

%config InlineBackend.figure_format = 'retina'
RANDOM_SEED = 8927
rng = np.random.default_rng(RANDOM_SEED)
az.style.use("arviz-darkgrid")
print(f"Running on PyMC v{pm.__version__}")

## Data

In [ ]:
raw = pd.read_csv("data/all-value-add-errors.csv")
names = pd.read_csv("data/school-names.csv").set_index("URN")

# ---- GCSE: six elements per school with known standard errors (schools with Progress 8 results)
elements = ["English", "Maths", "Science", "Humanities", "Languages", "Open"]
column = {"English": "P8MEAENG", "Maths": "P8MEAMAT", "Science": "SCIVAMEA_PTQ_EE",
          "Humanities": "HUMVAMEA_PTQ_EE", "Languages": "LANVAMEA_PTQ_EE", "Open": "P8MEAOPEN"}
frames = []
for e in elements:
    c = column[e]
    sub = raw[["URN", c, f"{c} lower", f"{c} upper"]].dropna()
    sub.columns = ["URN", "va", "lower", "upper"]
    sub["element"] = e
    frames.append(sub)
long = pd.concat(frames)
long["se"] = (long["upper"] - long["lower"]) / (2 * 1.96)
n_el = long.groupby("URN")["element"].nunique()
long = long[long["URN"].isin(n_el[n_el >= 3].index)].sort_values("URN").reset_index(drop=True)

# ---- all institutions: schools with GCSE results first, then institutions with A-level results only
urns_gcse = pd.Index(sorted(long["URN"].unique()))
urns_only = pd.Index(sorted(set(raw["URN"]) - set(urns_gcse)))
inst = urns_gcse.append(urns_only)
n_g, n_inst = len(urns_gcse), len(inst)
no_gcse = (np.arange(n_inst) >= n_g).astype(float)

long["school_idx"] = urns_gcse.get_indexer(long["URN"])
long["element_idx"] = long["element"].map({e: k for k, e in enumerate(elements)}).to_numpy()
n_elements = len(elements)
x_obs, x_se = long["va"].to_numpy(), long["se"].to_numpy()
s_idx, e_idx = long["school_idx"].to_numpy(), long["element_idx"].to_numpy()

region_series = raw.drop_duplicates("URN").set_index("URN")["RGN24NM"].reindex(inst)
regions = list(region_series.value_counts().index)
reg_idx = region_series.map({r: k for k, r in enumerate(regions)}).fillna(len(regions)).astype(int).to_numpy()   # unknown region -> national average
is_london = (region_series.to_numpy() == "London")
print(f"{n_g} schools with GCSE results, {len(urns_only)} institutions with A-level results only, {n_inst} in all; {is_london.sum()} in London")


# type of institution for those without GCSE results, and local authority for everyone
tg = names["type_group"].reindex(inst).to_numpy()
cat_names = ["independent school", "college", "other 16-19 / special"]
ng_cat = np.where(tg == "Independent schools", 0, np.where(tg == "Colleges", 1, 2))          # only used where no_gcse == 1
la_series = names["local_authority"].reindex(inst)
la_names = sorted(la_series.unique())
la_idx = la_series.map({a: k for k, a in enumerate(la_names)}).to_numpy()

In [ ]:
arts = ["Art & Design", "Art & Design (Fine Art)", "Art & Design (Photography)", "Art & Design (Graphics)", "Art & Design (Textiles)",
        "Art & Design (3d Studies)", "Art & Design (Critical Studies)", "Music", "Music Technology", "Drama & Theatre Studies", "Dance"]
groups = {"Maths": ["Mathematics"],
          "Economics": ["Economics"],
          "Psychology": ["Psychology"],
          "Other social sciences": ["Sociology", "Government & Politics", "Law"],
          "Sciences": ["Biology", "Chemistry", "Physics"],
          "English": ["English Literature", "English Language", "English Language & Literature"],
          "Humanities": ["History", "Geography", "Religious Studies", "Logic/ Philosophy", "Ancient History", "Classical Civilisation"],
          "Business & Computing": ["Business Studies:Single", "Computer Studies/Computing"],
          "Creative arts": arts}
group_names = list(groups)
n_groups = len(group_names)

raw_i = raw.set_index("URN").reindex(inst)
frames = []
for gname, subjects in groups.items():
    va = pd.DataFrame({s: raw_i[f"A-level {s} VA"] for s in subjects})
    se = pd.DataFrame({s: (raw_i[f"A-level {s} VA upper"] - raw_i[f"A-level {s} VA lower"]) / (2 * 1.96) for s in subjects})
    ent = pd.DataFrame({s: raw_i[f"A-level {s} entries"].where(va[s].notna(), 0).fillna(0) for s in subjects})
    total = ent.sum(axis=1)
    pooled_va = (va.fillna(0) * ent).sum(axis=1) / total.replace(0, np.nan)
    pooled_se = np.sqrt(((se.fillna(0) * ent) ** 2).sum(axis=1)) / total.replace(0, np.nan)   # entry-weighted; assumes separate cohorts
    f = pd.DataFrame({"inst_idx": np.arange(n_inst), "group": gname, "va": pooled_va.to_numpy(), "se": pooled_se.to_numpy(), "entries": total.to_numpy()})
    frames.append(f.dropna(subset=["va", "se"]))
alevel = pd.concat(frames).reset_index(drop=True)
alevel["group_idx"] = alevel["group"].map({g: k for k, g in enumerate(group_names)}).to_numpy()
y_obs, y_se = alevel["va"].to_numpy(), alevel["se"].to_numpy()
ys_idx, yg_idx = alevel["inst_idx"].to_numpy(), alevel["group_idx"].to_numpy()
assert (y_se > 0).all()
print(f"{len(alevel)} institution-group A-level observations")

### Choose the area

The read-out below uses this area. Set `AREA_REGION` to any region name (`"London"`, `"South East"`, ...), or leave it `None` and set `AREA_LA` to a local authority (`"Camden"`, ...). Everything before this is England-wide.

In [ ]:
AREA_REGION = "London"
AREA_LA = None
if AREA_LA is None:
    in_area, area_label = (region_series.to_numpy() == AREA_REGION), AREA_REGION
else:
    in_area, area_label = (la_series.to_numpy() == AREA_LA), AREA_LA
print(f"{area_label}: {in_area.sum()} institutions in the data")
area_obs = alevel[in_area[alevel["inst_idx"].to_numpy()]]
tab = area_obs.groupby("group").agg(institutions=("inst_idx", "nunique"), median_entries=("entries", "median"), median_SE=("se", "median"))
tab["England-wide institutions"] = alevel.groupby("group")["inst_idx"].nunique()
display(tab.loc[group_names].round(2))
display(names.loc[inst[in_area], "type_group"].value_counts().rename(f"{area_label} institutions").to_frame())

## Model

In [ ]:
keep = np.array([0.0 if e == "Humanities" else 1.0 for e in elements])
USE_LA = True

def build_model():
    coords = {"element": elements, "region": regions, "group": group_names, "cat": cat_names, "la": la_names}
    with pm.Model(coords=coords) as model:
        # ---- geography: region effects on GCSE quality, and (optionally) local-authority effects
        sigma_m = pm.HalfNormal("sigma_m", 0.5)
        m = pm.ZeroSumNormal("m", sigma=sigma_m, dims="region")
        m_all = pt.concatenate([m, pt.zeros(1)])
        g_loc = m_all[reg_idx]
        if USE_LA:
            sigma_a = pm.HalfNormal("sigma_a", 0.3)
            a_la = pm.Normal("a_la", 0, sigma_a, dims="la")         # local-authority shift in GCSE general quality, beyond region
            g_loc = g_loc + a_la[la_idx]

        # ---- GCSE side, schools with GCSE results only: general quality, tilt, consistency
        mu = pm.Normal("mu", 0, 1, dims="element")
        lam = pm.HalfNormal("lam", 1, dims="element")
        tau = pm.HalfNormal("tau", 0.5, dims="element")
        g = pm.Normal("g", g_loc[:n_g], 1, shape=n_g)
        h = pm.Normal("h", 0, 1, shape=n_g)
        k_raw = pm.Normal("kappa_raw", 0, 0.5, shape=n_elements)
        kappa = pm.Deterministic("kappa", k_raw * keep, dims="element")   # Humanities fixed at 0; the sign of the tilt is fixed after sampling
        sigma_s = pm.HalfNormal("sigma_s", 0.5)
        rho = pm.Deterministic("rho", 2 * pm.Beta("rho_raw", 2, 2) - 1)
        w = pm.Normal("w", 0, 1, shape=n_g)
        log_s = pm.Deterministic("log_s", sigma_s * (rho * (g - g_loc[:n_g]) + pt.sqrt(1 - rho**2) * w))
        pm.Normal("x_obs", mu=mu[e_idx] + lam[e_idx] * g[s_idx] + kappa[e_idx] * h[s_idx],
                  sigma=pt.sqrt((tau[e_idx] * pt.exp(log_s[s_idx]))**2 + x_se**2), observed=x_obs)

        # institutions without GCSE results: their GCSE quality is the typical one for where they are (no invented profile), no tilt
        g_full = pt.concatenate([g, g_loc[n_g:]])
        h_full = pt.concatenate([h, pt.zeros(n_inst - n_g)])

        # ---- A-level side (all institutions)
        nu = pm.Normal("nu", 0, 0.5, dims="group")
        sd_group = pm.HalfNormal("sd_group", 0.5, dims="group")           # group scatter, schools with GCSE results
        sd_group_ng = pm.HalfNormal("sd_group_ng", 0.5, dims="group")     # group scatter, institutions without
        sigma_psi = pm.HalfNormal("sigma_psi", 0.3)
        psi = pm.ZeroSumNormal("psi", sigma=sigma_psi, dims="region")
        psi_all = pt.concatenate([psi, pt.zeros(1)])
        shared_loc = psi_all[reg_idx]
        if USE_LA:
            sigma_xi = pm.HalfNormal("sigma_xi", 0.2)
            xi = pm.Normal("xi", 0, sigma_xi, dims="la")                    # local-authority shift in A-level shared quality, beyond region
            shared_loc = shared_loc + xi[la_idx]
        u = pm.Normal("u", 0, 1, shape=n_inst)
        b = pm.Normal("b", 0, 0.5, dims="group")
        c = pm.Normal("c", 0, 0.5, dims="group")
        lam_a = pm.HalfNormal("lam_a", 0.5, dims="group")
        d = pm.Normal("d", 0, 0.5, dims=("cat", "group"))                  # mean shift by type, institutions without GCSE results
        shared = shared_loc + u
        ng_cat_safe = np.maximum(ng_cat, 0)
        y_mean = (nu[yg_idx] + b[yg_idx] * g_full[ys_idx] + c[yg_idx] * h_full[ys_idx] + lam_a[yg_idx] * shared[ys_idx]
                  + d[ng_cat_safe[ys_idx], yg_idx] * no_gcse[ys_idx])
        sd_y = sd_group[yg_idx] * (1 - no_gcse[ys_idx]) + sd_group_ng[yg_idx] * no_gcse[ys_idx]
        pm.Normal("y_obs", mu=y_mean, sigma=pt.sqrt(sd_y**2 + y_se**2), observed=y_obs)
    return model

In [ ]:
model = build_model()

### Fit

In [ ]:
with model:
    idata = pm.sample(draws=1000, tune=2000, chains=4, target_accept=0.99, random_seed=RANDOM_SEED, progressbar=False)

### Diagnostics

In [ ]:
def align_sign(idata, tilt_terms):
    """The tilt is defined only up to sign; flip each chain to the orientation with Maths and Science positive and English and Open negative."""
    post = idata.posterior
    k = post["kappa"]
    score = (k.sel(element="Maths") + k.sel(element="Science") - k.sel(element="English") - k.sel(element="Open")).mean("draw")
    sign = xr.where(score > 0, 1.0, -1.0)
    out = post.copy()
    for name in tilt_terms:
        out[name] = post[name] * sign
    return out, sign.to_numpy()

post_a, sign_a = align_sign(idata, ["h", "kappa", "c"])
print("chains flipped:", int((sign_a < 0).sum()), "of", len(sign_a), "| divergences =", int(idata.sample_stats["diverging"].sum()))
skip = {"kappa_raw", "rho_raw"}
rh, es = az.rhat(post_a), az.ess(post_a)
worst = pd.DataFrame({"max r_hat": {v: float(rh[v].max()) for v in rh.data_vars if v not in skip},
                      "min bulk ESS": {v: float(es[v].min()) for v in es.data_vars if v not in skip}}).sort_values("max r_hat", ascending=False)
display(worst.round(3).head(8))

The weakest parameter is usually $\mu_e$, the overall GCSE level: it trades off against the mean of general quality, now including the local-authority shifts. It does not affect the estimates of institutions' value added, which depend on identified combinations.

## Geography

How much do regions and local authorities differ? $\sigma_m$ and $\sigma_a$ are the spreads of general GCSE quality between regions and between authorities within a region (in units of the within-authority spread of schools, which is 1). $\sigma_\psi$ and $\sigma_\xi$ are the corresponding spreads of the shared A-level quality that GCSE does not explain (in units where the institution-to-institution spread is 1; a group's own loading scales them into value-added points).

In [ ]:
def draws(name, thin=1):
    a = post_a[name].to_numpy()
    return a.reshape(-1, *a.shape[2:])[::thin]

for name, label in [("sigma_m", "regions, GCSE quality"), ("sigma_a", "local authorities, GCSE quality"),
                    ("sigma_psi", "regions, A-level shared quality"), ("sigma_xi", "local authorities, A-level shared quality")]:
    x = draws(name) if name != "sigma_xi" else draws("xi").std(axis=1)
    print(f"{name:10s} {label:44s} {x.mean():.3f} [{np.percentile(x, 5.5):.3f}, {np.percentile(x, 94.5):.3f}]")

a_la, xi = draws("a_la"), draws("xi")
la_in_area = np.array([np.any(in_area & (la_idx == k)) for k in range(len(la_names))])
tab = pd.DataFrame({"schools with GCSE": [int(np.sum((la_idx[:n_g] == k))) for k in range(len(la_names))],
                    "GCSE quality shift": a_la.mean(axis=0), "lo": np.percentile(a_la, 5.5, axis=0), "hi": np.percentile(a_la, 94.5, axis=0),
                    "A-level shared shift": xi.mean(axis=0), "lo ": np.percentile(xi, 5.5, axis=0), "hi ": np.percentile(xi, 94.5, axis=0)}, index=la_names)
display(tab[la_in_area].sort_values("GCSE quality shift", ascending=False).round(2))

## What the model says about the subjects

How each group's A-level value added relates to GCSE. $b_j$ and $c_j$ are the slopes on general GCSE quality and on the tilt (value-added points per SD); "GCSE explains" is the share of the group's true between-school variance carried by the two GCSE dimensions, among schools with GCSE results; the offsets are the mean shifts for institutions without GCSE results, by type.

In [ ]:
D = {k: draws(k, 5) for k in ["nu", "b", "c", "lam_a", "d", "sd_group", "sd_group_ng", "g", "h", "u", "m", "a_la", "xi", "psi"]}
n_draw = len(D["nu"])
m_all = np.concatenate([D["m"], np.zeros((n_draw, 1))], axis=1)
psi_all = np.concatenate([D["psi"], np.zeros((n_draw, 1))], axis=1)
g_loc = m_all[:, reg_idx] + D["a_la"][:, la_idx]                         # typical GCSE quality where each institution is
shared_loc = psi_all[:, reg_idx] + D["xi"][:, la_idx]
g_full = np.concatenate([D["g"], g_loc[:, n_g:]], axis=1)               # institutions without GCSE results: the typical one for where they are
h_full = np.concatenate([D["h"], np.zeros((n_draw, n_inst - n_g))], axis=1)

gs = slice(0, n_g)
comp = {"general GCSE quality": D["b"]**2 * g_full[:, gs].var(axis=1)[:, None], "tilt": D["c"]**2 * h_full[:, gs].var(axis=1)[:, None],
        "shared A-level residual": D["lam_a"]**2 * (shared_loc[:, gs] + D["u"][:, gs]).var(axis=1)[:, None], "group's own scatter": D["sd_group"]**2}
total = sum(comp.values())
summ = pd.DataFrame({"GCSE explains": ((comp["general GCSE quality"] + comp["tilt"]) / total).mean(axis=0),
                     "b (general quality)": D["b"].mean(axis=0), "c (tilt)": D["c"].mean(axis=0),
                     "shared A-level residual share": (comp["shared A-level residual"] / total).mean(axis=0),
                     "own scatter share": (comp["group's own scatter"] / total).mean(axis=0), "true SD": np.sqrt(total).mean(axis=0)}, index=group_names)
for k, cname in enumerate(cat_names):
    summ[f"offset: {cname}"] = D["d"][:, k, :].mean(axis=0)
display(summ.round(3))

## Estimating each institution's true value added

For a group $j$ and institution $i$ with observed value added $y^{obs}_{ij}$ and standard error $\sigma_{ij}$, and given one draw of the model's parameters, the true value added is the model's expectation for that institution (its GCSE profile if it has one, its shared A-level quality, its region and local authority, its type if it has no GCSE results) plus the group-specific departure $e_{ij}$. That departure has a known conditional distribution: the observed residual shrunk by $\sigma_j^2 / (\sigma_j^2 + \sigma_{ij}^2)$, so a small cohort (large $\sigma_{ij}$) is pulled hard toward the expectation and a large cohort barely at all. Repeating over posterior draws gives every institution a full distribution of its true value added.

In [ ]:
def true_va(group):
    """Posterior draws of the true A-level value added of every institution with a score in this group."""
    j = group_names.index(group)
    rows = alevel[alevel["group_idx"] == j]
    idx = rows["inst_idx"].to_numpy(); y, se = rows["va"].to_numpy(), rows["se"].to_numpy()
    gcse_part = D["b"][:, [j]] * g_full[:, idx] + D["c"][:, [j]] * h_full[:, idx]
    mean = (D["nu"][:, [j]] + gcse_part + D["lam_a"][:, [j]] * (shared_loc[:, idx] + D["u"][:, idx])
            + D["d"][:, np.maximum(ng_cat[idx], 0), j] * no_gcse[idx])
    sd2 = np.where(no_gcse[idx] == 1, D["sd_group_ng"][:, [j]], D["sd_group"][:, [j]]) ** 2
    k = sd2 / (sd2 + se**2)
    true = mean + k * (y - mean) + np.sqrt(k) * se * rng.standard_normal(mean.shape)
    return rows, true, gcse_part

subjects = ["Maths", "Economics", "Psychology"]
cache = {s: true_va(s) for s in subjects}

def shortlist(subject, state_funded_only=False):
    rows, true, gcse_part = cache[subject]
    idx = rows["inst_idx"].to_numpy()
    sel = in_area[idx]
    p_england = (true >= np.percentile(true, 80, axis=1)[:, None]).mean(axis=0)
    ta = true[:, sel]
    p_area = (ta >= np.percentile(ta, 80, axis=1)[:, None]).mean(axis=0)
    p_above_median = (ta >= np.median(ta, axis=1)[:, None]).mean(axis=0)
    r = rows[sel]
    urn = inst[r["inst_idx"].to_numpy()]
    df = pd.DataFrame({"URN": urn, "name": names.loc[urn, "name"].to_numpy(), "type": names.loc[urn, "type_group"].to_numpy(),
                       "borough": names.loc[urn, "local_authority"].to_numpy(), "postcode": names.loc[urn, "postcode"].to_numpy(),
                       "gender": names.loc[urn, "gender"].to_numpy(), "admissions": names.loc[urn, "admissions"].to_numpy(),
                       "entries": r["entries"].to_numpy(), "published VA": r["va"].to_numpy(), "SE": r["se"].to_numpy(),
                       "estimate": ta.mean(axis=0), "lo 89%": np.percentile(ta, 5.5, axis=0), "hi 89%": np.percentile(ta, 94.5, axis=0),
                       f"P(top fifth in {area_label})": p_area, "P(top fifth in England)": p_england[sel], f"P(above {area_label} median)": p_above_median,
                       "expected from GCSE": np.where(no_gcse[r["inst_idx"].to_numpy()] == 1, np.nan, gcse_part[:, sel].mean(axis=0))})
    if state_funded_only:
        df = df[df["type"] != "Independent schools"]
    return df.sort_values("estimate", ascending=False).reset_index(drop=True)

def show(subject, state_funded_only=False, top=50):
    df = shortlist(subject, state_funded_only)
    pa = f"P(top fifth in {area_label})"
    print(f"{subject}{' (state-funded only)' if state_funded_only else ''}: {len(df)} institutions in {area_label} with a score; "
          f"{(df[pa] > 0.8).sum()} are in its top fifth with probability above 0.8, {(df[pa] > 0.5).sum()} above 0.5")
    display(df.head(top).style.format(precision=3))
    return df

### Maths

In [ ]:
maths_all = show("Maths")

In [ ]:
maths_state = show("Maths", state_funded_only=True)

### Economics

In [ ]:
econ_all = show("Economics")

In [ ]:
econ_state = show("Economics", state_funded_only=True)

### Psychology

In [ ]:
psych_all = show("Psychology")

In [ ]:
psych_state = show("Psychology", state_funded_only=True)

### The top 50 at a glance

One chart per subject: the 50 institutions in the area with the highest estimated value added, best at the top. Each dot is the estimate and the line its 89% interval. **Marker size is the cohort** (entries) and **colour is the type**, so a big pale dot on a short line is well-evidenced and a small dot on a long line is not. The red cross is the published score; a long gap between a cross and its dot is the correction for a small cohort. The dotted line marks where London's top fifth starts (by estimate) and the dashed line is the national average (0). Files are saved to `results/`.

In [ ]:
import os
os.makedirs("results", exist_ok=True)
tag = area_label.lower().replace(" ", "-")
type_colour = {"state-funded": "#4C72B0", "independent": "#DD8452", "college": "#55A868"}
def short(name, k):
    return name if len(name) <= k else name[:k - 1].rstrip(" ,") + "…"

def kind_of(t):
    return "independent" if t == "Independent schools" else ("college" if t == "Colleges" else "state-funded")

def top_chart(subject, top=50):
    df = shortlist(subject)
    d = df.head(top).copy()
    d["kind"] = d["type"].map(kind_of)
    cut = np.percentile(df["estimate"], 80)
    y = np.arange(len(d))[::-1]
    fig, ax = plt.subplots(figsize=(11, 0.30 * len(d) + 2.2))
    for k, col in type_colour.items():
        m = (d["kind"] == k).to_numpy()
        if m.any():
            ax.hlines(y[m], d["lo 89%"].to_numpy()[m], d["hi 89%"].to_numpy()[m], color=col, linewidth=2.2, alpha=0.5)
            ax.scatter(d["estimate"].to_numpy()[m], y[m], s=14 + 7 * np.sqrt(d["entries"].to_numpy()[m]), color=col, edgecolor="white", linewidth=0.6, zorder=3, label=k)
    ax.scatter(d["published VA"], y, marker="x", color="#C44E52", s=26, zorder=4, label="published score")
    ax.axvline(0, color="grey", linewidth=0.9, linestyle="--")
    ax.axvline(cut, color="grey", linewidth=0.9, linestyle=":")
    ax.text(cut, len(d) - 0.3, " top fifth\n starts", fontsize=8, color="grey", va="bottom")
    labels = [f"{short(n, 38)}  ({int(e)})" for n, e in zip(d["name"], d["entries"])]
    ax.set_yticks(y, labels, fontsize=8)
    ax.set_ylim(-1, len(d) + 1)
    ax.set_xlabel(f"{subject} A-level value added (points); label shows number of entries")
    ax.set_title(f"{subject}: top {len(d)} in {area_label}", loc="left")
    ax.legend(loc="lower right", fontsize=8, frameon=True)
    plt.tight_layout()
    plt.savefig(f"results/{tag}-{subject.lower()}-top{top}.png", dpi=150, bbox_inches="tight")
    plt.show()
top_chart("Maths")

In [ ]:
top_chart("Economics")

In [ ]:
top_chart("Psychology")

### Outliers: published against estimated

Every institution in the area with a score. The dashed diagonal is "no correction". Points **below the diagonal on the right** are high published scores the model does not believe on that evidence, almost always small cohorts (small dots); points **above it on the left** are the reverse. Points close to the diagonal have cohorts large enough that the published score is taken at face value. The ten biggest corrections and the three best estimates are labelled in the column at the right, with the number of entries in brackets.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 7.5))
for ax, subject in zip(axes, subjects):
    df = shortlist(subject)
    df["kind"] = df["type"].map(kind_of)
    df["correction"] = df["published VA"] - df["estimate"]
    for k, col in type_colour.items():
        m = df["kind"] == k
        ax.scatter(df.loc[m, "published VA"], df.loc[m, "estimate"], s=10 + 6 * np.sqrt(df.loc[m, "entries"]), color=col, alpha=0.55, edgecolor="white", linewidth=0.4, label=k)
    lo_l = min(df["published VA"].min(), df["estimate"].min()) - 0.1
    hi_l = max(df["published VA"].max(), df["estimate"].max()) + 0.1
    ax.plot([lo_l, hi_l], [lo_l, hi_l], color="grey", linewidth=0.9, linestyle="--")
    ax.axhline(0, color="lightgrey", linewidth=0.7); ax.axvline(0, color="lightgrey", linewidth=0.7)
    # label the biggest corrections in a column at the right, joined to their points
    chosen = pd.concat([df.sort_values("correction", ascending=False).head(6), df.sort_values("correction").head(4), df.head(3)]).drop_duplicates("URN")
    chosen = chosen.sort_values("estimate", ascending=False)
    span = hi_l - lo_l
    x_text = hi_l + 0.04 * span
    y_slots = np.linspace(hi_l, lo_l + 0.25 * span, len(chosen))
    for (_, r), ys in zip(chosen.iterrows(), y_slots):
        ax.annotate(f"{short(r['name'], 30)} ({int(r['entries'])})", (r["published VA"], r["estimate"]), xytext=(x_text, ys), textcoords="data", fontsize=7.5, va="center",
                    arrowprops=dict(arrowstyle="-", color="grey", linewidth=0.5, shrinkA=0, shrinkB=2), annotation_clip=False)
    ax.set_xlim(lo_l, hi_l + 0.62 * span)
    ax.set_xlabel("published value added"); ax.set_ylabel("estimated true value added")
    ax.set_title(f"{subject}, {area_label}", loc="left")
axes[0].legend(fontsize=8, loc="upper left", title="marker size = cohort", title_fontsize=8)
plt.tight_layout()
plt.savefig(f"results/{tag}-published-vs-estimate.png", dpi=150, bbox_inches="tight")
plt.show()

### What the correction does

Ranking institutions on the published number and on the model's estimate give different lists, mainly because small cohorts give extreme published scores. For each subject: how many of the published top ten remain in the estimate's top ten, and how large the cohorts are in each list.

In [ ]:
rows_out = []
for subject in subjects:
    df = shortlist(subject)
    raw_top, est_top = df.sort_values("published VA", ascending=False).head(10), df.head(10)
    rows_out.append({"subject": subject, f"{area_label} institutions": len(df), "top ten in both": len(set(raw_top["URN"]) & set(est_top["URN"])),
                     "median entries, published top ten": raw_top["entries"].median(), "median entries, estimate top ten": est_top["entries"].median(),
                     "median SE, published top ten": raw_top["SE"].median(), "median SE, estimate top ten": est_top["SE"].median()})
pd.DataFrame(rows_out).set_index("subject").round(2)

### Strong in more than one of the three

Institutions whose probability of being in the area's top fifth is above 0.8 in at least two of the three subjects, with their estimate, cohort and probability in each. (A blank means the institution has no score for that subject.)

In [ ]:
pa = f"P(top fifth in {area_label})"
frames = []
for s in subjects:
    df = shortlist(s)[["URN", "name", "type", "borough", "entries", "estimate", pa]]
    frames.append(df.set_index("URN").rename(columns={"entries": f"entries [{s}]", "estimate": f"estimate [{s}]", pa: f"P(top fifth) [{s}]", "name": f"name [{s}]",
                                                       "type": f"type [{s}]", "borough": f"borough [{s}]"}))
both = pd.concat(frames, axis=1)
p_cols = [f"P(top fifth) [{s}]" for s in subjects]
both["subjects with P > 0.8"] = (both[p_cols] > 0.8).sum(axis=1)
both["name"] = both[[f"name [{s}]" for s in subjects]].bfill(axis=1).iloc[:, 0]
both["type"] = both[[f"type [{s}]" for s in subjects]].bfill(axis=1).iloc[:, 0]
both["borough"] = both[[f"borough [{s}]" for s in subjects]].bfill(axis=1).iloc[:, 0]
show_cols = ["name", "type", "borough", "subjects with P > 0.8"] + [c for s in subjects for c in (f"estimate [{s}]", f"P(top fifth) [{s}]", f"entries [{s}]")]
multi = both[both["subjects with P > 0.8"] >= 2].sort_values(["subjects with P > 0.8", "name"], ascending=[False, True])
print(f"{(both[p_cols].notna().sum(axis=1) == 3).sum()} institutions in {area_label} have a score in all three subjects; {len(multi)} are strong (P > 0.8) in at least two")
display(multi[show_cols].style.format(precision=2))

### Export

The full shortlists for the chosen area, all institutions and subjects, are written to `results/` so they can be sorted and filtered outside the notebook.

In [ ]:
import os
os.makedirs("results", exist_ok=True)
tag = area_label.lower().replace(" ", "-")
for s in subjects:
    shortlist(s).to_csv(f"results/{tag}-{s.lower()}-shortlist.csv", index=False)
both[show_cols].to_csv(f"results/{tag}-three-subjects.csv")
print("written:", sorted(os.listdir("results")))

## Do the estimates pull in the tails? Published against estimated, all of England

The shortlist tables and charts above are for one area. This section asks the general question for **the whole of England and every subject group in the model**: how does the estimated true value added relate to the published value added, and is the correction larger in the tails?

Two things to read:

- The **slope of the estimate on the published score** (a slope of 1 would mean the estimate follows the published score; below 1 means the estimates are pulled toward the average, so extreme published scores become less extreme).
- The **size of the correction** (estimate minus published) in the tails of the published scores compared with the middle. Because an institution's published score is noisy, and the noise is largest for small cohorts, the extremes of the published scores are mostly small cohorts, and those are the ones the model pulls in.

The table gives, for each subject group in England (and the slope for London alone): the slope, the ratio of the spread of the estimates to the spread of the published scores, and for the bottom and top 5% of published scores their average published score, average estimate, and the typical size of the correction compared with the middle half; plus how large the cohorts in the tails are.

In [ ]:
rows = []
store = {}
for gname in group_names:
    rows_, true, _ = true_va(gname)
    pub, ent = rows_["va"].to_numpy(), rows_["entries"].to_numpy()
    est = true.mean(axis=0)
    idx = rows_["inst_idx"].to_numpy()
    lon = region_series.to_numpy()[idx] == "London"
    corr = est - pub
    lo5, hi5 = pub <= np.percentile(pub, 5), pub >= np.percentile(pub, 95)
    mid = (pub >= np.percentile(pub, 25)) & (pub <= np.percentile(pub, 75))
    store[gname] = (pub, est, ent)
    rows.append({"group": gname, "institutions": len(pub),
                 "slope, England": np.polyfit(pub, est, 1)[0], "slope, London": np.polyfit(pub[lon], est[lon], 1)[0],
                 "spread of estimates / spread of published": est.std() / pub.std(),
                 "bottom 5%: published": pub[lo5].mean(), "bottom 5%: estimate": est[lo5].mean(),
                 "top 5%: published": pub[hi5].mean(), "top 5%: estimate": est[hi5].mean(),
                 "typical |correction|, tails": np.abs(corr[lo5 | hi5]).mean(), "typical |correction|, middle half": np.abs(corr[mid]).mean(),
                 "median entries, tails": np.median(ent[lo5 | hi5]), "median entries, all": np.median(ent),
                 "share with |estimate| > |published|": (np.abs(est) > np.abs(pub)).mean()})
tail_tab = pd.DataFrame(rows).set_index("group")
display(tail_tab.round(3))

### The pattern, by subject group

Each panel bins the institutions in England by their published score (20 bins of equal size) and shows the average estimate in each bin, with a band of one standard deviation, against the diagonal ("no correction"). Where the line lies flatter than the diagonal, extreme published scores are being pulled in. The slope for the group is shown in the panel.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 13)); axes = axes.ravel()
for ax, gname in zip(axes, group_names):
    pub, est, ent = store[gname]
    bins = np.quantile(pub, np.linspace(0, 1, 21)); which = np.clip(np.digitize(pub, bins[1:-1]), 0, 19)
    bx = np.array([pub[which == b].mean() for b in range(20)]); by = np.array([est[which == b].mean() for b in range(20)]); bs = np.array([est[which == b].std() for b in range(20)])
    ax.scatter(pub, est, s=4, color="#BBBBBB", alpha=0.35, zorder=1)
    ax.fill_between(bx, by - bs, by + bs, color="#4C72B0", alpha=0.2, zorder=2)
    ax.plot(bx, by, "-o", color="#4C72B0", markersize=4, zorder=3)
    lim = (min(pub.min(), est.min()) - 0.05, max(pub.max(), est.max()) + 0.05)
    ax.plot(lim, lim, color="grey", linewidth=0.9, linestyle="--")
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_title(f"{gname}: slope {np.polyfit(pub, est, 1)[0]:.2f}", loc="left")
    ax.set_xlabel("published value added"); ax.set_ylabel("estimated true value added")
plt.tight_layout()
plt.savefig("results/tail-shrinkage-england.png", dpi=140, bbox_inches="tight")
plt.show()

## Summary

Fitted with 0 divergences. The weakest parameter is the overall GCSE level $\mu_e$ ($\hat R$ 1.10, ESS 51), which does not affect the institution estimates.

**What GCSE tells you about each of the three subjects**

| | Maths | Economics | Psychology |
| --- | --- | --- | --- |
| Share of true between-school variance explained by GCSE | 28% | 8.5% | 8.5% |
| Slope on general GCSE quality (points per SD) | 0.095 | 0.089 | 0.086 |
| Slope on the tilt (Maths-and-Science side positive) | +0.18 | +0.03 | -0.05 |
| Shared A-level quality (not GCSE) | 49% | 53% | 45% |
| The subject's own scatter | 23% | 39% | 46% |

- **For Maths, a school's GCSE profile is informative**, mainly through the tilt: a school leaning to Maths and Science at GCSE tends to do well in A-level Maths. **For Economics and Psychology it says little** (8.5% each), so those estimates rest on the institution's own A-level results and its other subjects, and are less certain.
- **Geography below region is real.** Local authorities differ in general GCSE quality by SD 0.31 beyond region (regions: 0.35), and in the shared A-level quality by SD 0.19; the regional spread of the latter shrinks to 0.08 once authorities are in the model. So where an institution sits matters at borough level, and the estimates shrink toward the local, not just the regional, picture.
- **Institutions without GCSE results** (independent schools, colleges) sit at different levels: independent schools about +0.15 (Psychology), +0.16 (Maths) and +0.29 (Economics) above state schools with a comparable profile; colleges about -0.12 to -0.26 below. These are averages for the type, applied when we have no other information, and they set where small cohorts are pulled to.

**The London read-out.** 469 London institutions have a Maths score, 397 Economics and 422 Psychology. Of these, 40, 28 and 33 respectively are in London's top fifth with probability above 0.8 (34, 12 and 29 among state-funded institutions). Only 5 are that strong in all three subjects (Ark Isaac Newton Academy, JFS, City of London Academy Highgate Hill, Kneller Hall School and Mill Hill Schools) and 16 in at least two, listed above. The correction for cohort size matters: only 6 to 7 of the published top ten in each subject stay in the estimate's top ten, and the published top tens have small cohorts (median 10-25 entries) against 14-39 for the estimate's.

**Reading the tables**

- **Prefer evidence over rank.** An institution with a probability near 1.0 and a large cohort (for example Brampton Manor Academy in Maths, 280 entries; Drayton Manor, Featherstone and Avanti House, 50-85; Ark Isaac Newton Academy and JFS across all three subjects) is a much stronger finding than one with a high estimate on 10 entries and an interval as wide as the gap between good and average.
- **Economics has the least evidence:** only 12 state-funded institutions reach 0.8, and many of the high estimates are on 10 to 20 entries.
- **Selection is not modelled.** Value added compares pupils with similar prior attainment, but who chooses the subject and who is admitted differs. The two Maths schools at the top of the Maths list (Imperial College London Mathematics School and King's College London Maths School) are specialist sixth forms that, to my understanding, admit at 16 on maths aptitude, and several independent schools are selective; the register's admissions field does not capture either fully. Some schools are single-sex (the tables show the gender in the register for those who need it).
- **Nothing here says whether a sixth form takes external applicants or what grades it requires.** Check that separately for any shortlisted school.
- **One year of results.** Small changes between years are noise; treat the tables as a shortlist to investigate, not a ranking to follow.

**Published against estimated, all of England.** In every subject group the estimates are pulled *inward* relative to the published scores, not pushed outward, and the correction is far larger in the tails.

- **Slope of the estimate on the published score** (England): Maths 0.64, Economics 0.66, Psychology 0.68, Other social sciences 0.60, Sciences 0.70, English 0.55, Humanities 0.59, Business & Computing 0.59, Creative arts 0.67; London gives similar slopes (0.53 to 0.74). The spread of the estimates is 60% to 75% of the spread of the published scores.
- **In the tails:** the bottom 5% of published Maths scores average $-1.28$ and are estimated at $-0.83$; the top 5% average $+0.86$ and are estimated at $+0.48$. Economics: $-1.04$ to $-0.66$ and $+0.80$ to $+0.51$. Psychology: $-1.03$ to $-0.71$ and $+0.82$ to $+0.51$.
- **The correction is 4 to 6 times larger in the tails** than in the middle half (0.31 to 0.43 points in the tails against 0.065 to 0.09 in the middle), and the tails are dominated by small cohorts (median 9 to 25 entries against 14 to 43 overall).
- **In 73% to 84% of institutions the estimate is closer to zero than the published score**; it is more extreme in only 16% to 27%, mostly where a large cohort's published score is being moved slightly by the model's expectation from its other information.
- **Which groups are pulled most:** English (0.55), Humanities and Business & Computing (0.59) and Other social sciences (0.60); least in Sciences (0.70), which has the largest cohorts. Maths, Economics and Psychology sit in the middle (0.64 to 0.68).
- **Why:** a published score is a noisy measure of an institution's true value added, and the noise is largest for small cohorts. Partial pooling shrinks each score toward what the institution's other information supports, more when the score is noisy, so extreme published scores, which are disproportionately noisy small-cohort ones, move most. Slopes below 1 are what this method should give; how far below depends on how much of the spread in published scores is noise. For the pooled groups the pooled standard errors are a lower bound, which would, if anything, understate the noise and the pull.
